In [1]:
import paddle
from paddle.io import DataLoader, random_split
from paddle.vision.datasets import Cifar10
from darknet import Darknet53

C:\Users\yvanw\miniconda3\envs\paddle\Lib\site-packages\paddle\utils\cpp_extension\extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)


In [2]:
transform = paddle.vision.transforms.Compose([
    paddle.vision.transforms.ToTensor(),
    paddle.vision.transforms.Resize((224, 224)),
    paddle.vision.transforms.Normalize(mean=[0.4914, 0.4822, 0.4465], std=[0.2470, 0.2435, 0.2616])
])

mnist = Cifar10(mode='train', backend='cv2', transform=transform)
mnist_test = Cifar10(mode='test', backend='cv2', transform=transform)

mnist_train, mnist_val = random_split(mnist, [round(0.8*len(mnist)), round(0.2*len(mnist))])

In [3]:
train_loader = DataLoader(mnist_train, batch_size=128, shuffle=True)
val_loader = DataLoader(mnist_val, batch_size=128, shuffle=False)
test_loader = DataLoader(mnist_test, batch_size=128, shuffle=False)

# plt.imshow(list(train_loader)[0][0][4].numpy().reshape([28, 28]))

img, label = next(iter(train_loader))
img.shape, label.shape

(paddle.Size([128, 3, 224, 224]), paddle.Size([128]))

In [4]:
model = Darknet53()
model

Darknet53(
  (features): Sequential(
    (0): BasicConvLayer(
      (conv): Conv2D(3, 32, kernel_size=[3, 3], padding=1, data_format=NCHW)
      (bn): BatchNorm2D(num_features=32, momentum=0.9, epsilon=1e-05)
      (act): LeakyReLU(negative_slope=0.1)
    )
    (1): ResidualXBlock(
      (downSample): BasicConvLayer(
        (conv): Conv2D(32, 64, kernel_size=[3, 3], stride=[2, 2], padding=1, data_format=NCHW)
        (bn): BatchNorm2D(num_features=64, momentum=0.9, epsilon=1e-05)
        (act): LeakyReLU(negative_slope=0.1)
      )
      (residual1_1): ResidualBlock(
        (cbl1): BasicConvLayer(
          (conv): Conv2D(64, 32, kernel_size=[1, 1], data_format=NCHW)
          (bn): BatchNorm2D(num_features=32, momentum=0.9, epsilon=1e-05)
          (act): LeakyReLU(negative_slope=0.1)
        )
        (cbl2): BasicConvLayer(
          (conv): Conv2D(32, 64, kernel_size=[3, 3], padding=1, data_format=NCHW)
          (bn): BatchNorm2D(num_features=64, momentum=0.9, epsilon=1e-05)
   

In [5]:
paddle.summary(model, (1, 3, 224, 224))

-------------------------------------------------------------------------------
   Layer (type)         Input Shape          Output Shape         Param #    
    Conv2D-104       [[1, 3, 224, 224]]   [1, 32, 224, 224]         864      
  BatchNorm2D-104   [[1, 32, 224, 224]]   [1, 32, 224, 224]         128      
   LeakyReLU-104    [[1, 32, 224, 224]]   [1, 32, 224, 224]          0       
BasicConvLayer-104   [[1, 3, 224, 224]]   [1, 32, 224, 224]          0       
     Conv2D-53      [[1, 32, 224, 224]]   [1, 64, 112, 112]       18,432     
  BatchNorm2D-53    [[1, 64, 112, 112]]   [1, 64, 112, 112]         256      
   LeakyReLU-53     [[1, 64, 112, 112]]   [1, 64, 112, 112]          0       
 BasicConvLayer-53  [[1, 32, 224, 224]]   [1, 64, 112, 112]          0       
     Conv2D-54      [[1, 64, 112, 112]]   [1, 32, 112, 112]        2,048     
  BatchNorm2D-54    [[1, 32, 112, 112]]   [1, 32, 112, 112]         128      
   LeakyReLU-54     [[1, 32, 112, 112]]   [1, 32, 112, 112]   

{'total_params': 40630890, 'trainable_params': 40595178}

In [ ]:
optimizer = paddle.optimizer.AdamW(learning_rate=0.01, parameters=model.parameters(), weight_decay=0.001)
loss_fn = paddle.nn.CrossEntropyLoss()

In [ ]:
model.train()
for epoch in range(5):
    for batch_id, (images, labels) in enumerate(train_loader):        
        logits = model(images)               # [64, 100, 1, 1]
        logits = logits.squeeze()       
        loss = loss_fn(logits, labels)        
        loss.backward()
        optimizer.step()
        optimizer.clear_grad()
        if batch_id % 30 == 0:
            print(f'Epoch: {epoch}, Batch_id: {batch_id}, Loss: {loss.item()}')
    model.eval()
    correct, total = 0, 0
    with paddle.no_grad():
        for images, labels in val_loader:
            logits = model(images)
            preds = paddle.argmax(logits, axis=1)
            correct += (preds == labels).numpy().sum()
            total += labels.shape[0]    
    acc = correct / total
    print(f"Epoch {epoch} Finished, Test Acc: {acc:.4f}")
    model.train()

In [ ]:
from utils import save_weight

save_weight(model, './darknet')